In [2]:
!pip install openai --quiet

# **Review Summarization Using Generative AI**

- **Objective:** Summarize reviews into articles that recommend the top products for each category.

- **Task:** Create a model that generates a short article (like a blog post) for each product category. The output should include:

    - Top 3 products and key differences between them.
    - Top complaints for each of those products.
    - Worst product in the category and why it should be avoided.

## Setup:

In [18]:
# Import Required Packages
import openai
import numpy as np
from datasets import load_dataset
import pandas as pd

In [4]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [5]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)


def get_completion(prompt, model="gpt-3.5-turbo"): # Andrew mentioned that the prompt/ completion paradigm is preferable for this class
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content


## Load the datasets

- merged_df: contains columns ['parent_asin','title','text','rating','meta_category']
- meta_df:   contains columns ['parent_asin','title']


In [7]:
# Load the raw reviews dataset for All Beauty from Hugging Face
raw_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                           "raw_review_All_Beauty", 
                           trust_remote_code=True)


In [8]:
# Load the Item Metadata Dataset
# Using the "raw_meta_All_Beauty" configuration to obtain product-related information.
meta_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                            "raw_meta_All_Beauty", 
                            split="full", 
                            trust_remote_code=True)

# Convert both datasets to Pandas DataFrames for easier merging
reviews_df = raw_dataset["full"].to_pandas()  # your reviews dataset already loaded earlier
meta_df = meta_dataset.to_pandas()

In [9]:
print("reviews df columns are:",reviews_df.columns)
print("meta df columns are:",meta_df.columns)

reviews df columns are: Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')
meta df columns are: Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
      dtype='object')


In [12]:
# Merge on "parent_asin" to combine review info with metadata (which contains, for instance, the "store" field)
merged_df = pd.merge(
    reviews_df[["parent_asin", "text", "rating"]],
    meta_df[["parent_asin", "title", "main_category"]],
    on="parent_asin",
    how="inner"
)

In [23]:
meta_df.to_csv("mete_df.csv", index=False)

In [14]:
for meta_cat in merged_df["main_category"].dropna().unique():
    # 1) Filter reviews for this meta‑category
    cat_reviews = merged_df[merged_df["main_category"] == meta_cat]
    
    # 2) Compute average rating per product
    avg_ratings = (
        cat_reviews
        .groupby("parent_asin")["rating"]
        .mean()
        .astype(float)
    )
    
    # 3) Top 3 products by highest average rating
    top3 = avg_ratings.sort_values(ascending=False).head(3).index.tolist()
    
    # 4) Worst product by lowest average rating
    worst = avg_ratings.idxmin()
    
    # 5) Build the “Input Data” section for the prompt
    sections = []
    for asin in top3:
        # Get product title directly from merged_df
        title = cat_reviews[cat_reviews["parent_asin"] == asin]["title"].iloc[0]
        sample = cat_reviews[cat_reviews["parent_asin"] == asin]["text"]\
             .sample(n=min(3, len(cat_reviews[cat_reviews["parent_asin"] == asin])), random_state=42)

        reviews_md = "\n".join(f'- \"{r}\"' for r in sample)
        sections.append(f"Product: {title} (ASIN: {asin})\nReviews:\n{reviews_md}")
    
    # 6) Add worst product samples
    w_title = cat_reviews[cat_reviews["parent_asin"] == worst]["title"].iloc[0]
    w_sample = cat_reviews[cat_reviews["parent_asin"] == worst]["text"]\
               .sample(n=min(2, len(cat_reviews[cat_reviews["parent_asin"] == worst])), random_state=42)

    w_reviews_md = "\n".join(f'- \"{r}\"' for r in w_sample)
    sections.append(f"Worst Product: {w_title} (ASIN: {worst})\nReviews:\n{w_reviews_md}")
    


In [15]:
# 7) Assemble and send your prompt to the model...
prompt = f"""Generate a short blog‑style recommendation article for the category: **{meta_cat}**.
            Input Data:
            {chr(10).join(sections)}

            Instructions:
                1. Title: “Top Picks for {meta_cat}”
                2. List the Top 3 products with:
                - Their names and key differences.
                3. Under each product, bullet‑list the Top 2 complaints from its reviews.
                4. Finally, name the Worst Product and explain why it should be avoided.
                5. Keep the total length to around 200–250 words.
                6. Use a friendly, informative tone.

                Produce the article in plain text with headings and bullets.
                """


In [ ]:
# Make chat request
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a professional product review writer."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.7,
    max_tokens=500,
    top_p=0.9
)

# Extract result
article = response.choices[0].message.content.strip()


In [21]:
print(article)

**Top Picks for Premium Beauty**

1. **L'ANZA Keratin Healing Oil Lustrous Shine Spray**
   - Key Features: Heat protectant spray for hair that tames frizz, eliminates flyaways, and adds a sparkling finish and silky feel.
   - Top Complaints:
     - "Love this product"
     - "Love all their products"

2. **FOREO LUNA 4 mini Face Cleansing Brush & Face Massager**
   - Key Features: Premium face care tool that enhances absorption of facial skin care products, suitable for all skin types.
   - Top Complaints:
     - "I absolutely love this! Not only is its design unique and aesthetically pleasing, it feels great on the skin and cleans well."

3. **MakeUp Eraser, 7-Day Set**
   - Key Features: Set of makeup erasers that remove all makeup with just water, including waterproof mascara, eyeliner, foundation, lipstick, and more.
   - Top Complaints:
     - "They are pretty as well as soft and DO remove makeup - even eye makeup. They launder perfectly in the little bag that came with the set."